In [6]:
import sys
!{sys.executable} -m pip install eli5


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import eli5
from eli5.sklearn import PermutationImportance

# 1. Load the dataset directly
# (You can also fetch directly from MySQL using sqlalchemy, but reading the local CSV is faster)
df = pd.read_csv('churn-bigml-20.csv')

# 2. Preprocessing & Binary Conversions
# Convert categorical string dimensions into binary indicators
df['International plan'] = df['International plan'].map({'Yes': 1, 'No': 0})
df['Voice mail plan'] = df['Voice mail plan'].map({'Yes': 1, 'No': 0})
df['Churn'] = df['Churn'].astype(int)

# Drop high-cardinality geographic non-numeric columns for the ML model matrix
X = df.drop(columns=['State', 'Area code', 'Churn'])
y = df['Churn']

# 3. Stratified Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Model Training (Random Forest Classifier)
print("Training the binary classification model...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. Model Evaluation
y_pred = model.predict(X_test)
print("\n--- Model Evaluation Diagnostics ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

# ========================================================
# 6. MODEL EXPLAINABILITY WITH ELI5
# ========================================================
print("\n--- Computing Global Feature Weights via ELI5 ---")
# Show weights based on the structural split logic inside the trees
display(eli5.show_weights(model, feature_names=list(X.columns)))

# Compute Permutation Importance as requested by your guide
perm = PermutationImportance(model, random_state=42).fit(X_test, y_test)
print("\n--- Permutation Feature Importance Weights ---")
display(eli5.show_weights(perm, feature_names=list(X.columns)))

Training the binary classification model...

--- Model Evaluation Diagnostics ---
Accuracy Score: 92.54%

--- Confusion Matrix ---
[[115   0]
 [ 10   9]]

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.92      1.00      0.96       115
           1       1.00      0.47      0.64        19

    accuracy                           0.93       134
   macro avg       0.96      0.74      0.80       134
weighted avg       0.93      0.93      0.91       134


--- Computing Global Feature Weights via ELI5 ---


Weight,Feature
0.1361 ± 0.1636,Total day charge
0.1335 ± 0.1827,Total day minutes
0.1051 ± 0.0867,Customer service calls
0.0937 ± 0.1043,Total eve charge
0.0861 ± 0.1064,Total eve minutes
0.0446 ± 0.0587,Total intl charge
0.0424 ± 0.0563,Total eve calls
0.0409 ± 0.0632,International plan
0.0408 ± 0.0621,Total intl minutes
0.0405 ± 0.0626,Total day calls



--- Permutation Feature Importance Weights ---


Weight,Feature
0.0403 ± 0.0152,Total day minutes
0.0388 ± 0.0304,Customer service calls
0.0373 ± 0.0094,Total day charge
0.0194 ± 0.0073,Number vmail messages
0.0104 ± 0.0202,Voice mail plan
0.0090 ± 0.0060,Total intl calls
0.0075 ± 0.0094,Total night charge
0.0075 ± 0.0000,Total night calls
0.0075 ± 0.0163,International plan
0.0060 ± 0.0112,Total day calls


In [9]:
import eli5
from eli5.formatters import format_as_html

# Export the feature weights table as a clean web page file
html_content = format_as_html(eli5.explain_weights(model, feature_names=list(X.columns)))

with open("eli5_weights_report.html", "w") as f:
    f.write(html_content)
    
print("Table successfully exported! Go to your project folder and double-click 'eli5_weights_report.html' to open it in Chrome/Edge with perfect visibility.")

Table successfully exported! Go to your project folder and double-click 'eli5_weights_report.html' to open it in Chrome/Edge with perfect visibility.
